# Trabajo Fin de Máster  
### Análisis de la Ciudad mediante Aprendizaje Supervisado  
#### Detección Automática de Tipologías Residenciales y Patrones de Cerramiento: Interpretabilidad vs Rendimiento

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Metodología para el análisis de interpretabilidad

### 📝 Descripción del notebook
TO COMPLETE...

### Indice de contenidos
TO COMPLETE...

# Configuración de entorno e *imports*

Este proyecto ha sido realizado en un entorno Anaconda con la versión 3.11.15 de *Python*. Las versiones de las librerias requeridas se encuentran en el fichero *requirements.txt*.

En esta sección se importan las librerias necesarias para la ejecución de este fichero jupyter notebook, se activa el *reload* de módulos externos y se configuran aspectos globales y de reproducibilidad.

In [1]:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from src.balanced_xgb import BalancedXGBClassifier

# Configuration object
from src.config import cfg

# Production training method
from src.production_training import train_final_model

**Import troubleshooting**

If the `src` imports fail when running this notebook in a different environment,
uncomment and execute the following cell:

```python
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
# Global configuration
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["font.family"] = "sans-serif"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

In [4]:
# Reproducibility
SEED = cfg.SEED 
np.random.seed(SEED)
random.seed(SEED)

# 1. Análisis de interpretabilidad global

# 1.1 Entrenamiento de modelos sobre todo el conjunto de datos

En la fase de evaluación de rendimiento se ha empleado una estrategia de *Nested Cross Validation*  con el objetivo de estimar de forma realista la capacidad de generalización de los distintos algoritmos. Este enfoque permite evaluar cómo se comporta un determinado algoritmo al aprender de un conjunto de entrenamiento y generalizar ante datos no observados, simulando el escenario de producción, en el que el modelo se entrena con la información disponible hasta un determinado momento y realiza predicciones sobre los nuevos datos que ingresan al sistema.

El objetivo del análisis de interpretabilidad, sin embargo, es diferente. En este caso, no se pretende estimar la capacidad de generalización, sino comprender el comportamiento global aprendido por el modelo a partir de los datos. Por este motivo, la interpretabilidad se estudia a partir de modelos entrenados sobre todo el conjunto de datos disponible, maximizando la información utilizada para capturar la estructura subyacente del problema.
De esta forma se pretende explicar el modelo final que se desplegaría en producción, el cual se entrena con todos los datos históricos disponibles.

En la *Nested Cross Validation* se obtiene un modelo óptimo para cada partición externa, donde cada uno de ellos se configura con un conjunto de hiperparámetros optimizados específicamente para el subconjunto de entrenamiento correspondiente a dicha partición. Como se ha mencionado previamente, el análisis de interpretabilidad requiere disponer de un único modelo final. Una alternativa natural para la selección de hiperparámetros consistiría en utilizar la moda de los hiperparámetros obtenidos en las distintas particiones de la *Nested Cross Validation*. Sin embargo, este enfoque no garantiza la optimalidad global, ni necesariamente refleja la mejor configuración sobre el conjunto completo de datos.

Para obtener la configuración óptima sobre el conjunto completo de datos, se realiza una búsqueda de hiperparámetros utilizando el mismo espacio de búsqueda definido en la fase de evaluación de rendimiento. Esta estrategia corresponde al procedimiento estándar en el despliegue de modelos en producción, donde el objetivo es optimizar el rendimiento del modelo utilizando toda la información disponible hasta el momento.

Para llevar a cabo esta optimización, se aplica una validación cruzada estándar que utiliza exactamente las mismas particiones empleadas en el bucle externo (*Outer Loop*) de la *Nested Cross Validation*, garantizando así la reprodubilidad en la construcción de los modelos finales.

A continuación se entrenan todos los modelos mediante la funcion *train_final_model()* ubicada en el módulo *src/production_training.py*. Los modelos se almacenan en el directorio *output/models* en formato .pkl.

In [5]:
################################
# Multinomial Logistic Regression
################################

LR_model = LogisticRegression(
    class_weight="balanced",
    solver="lbfgs",
    penalty="l2",
    random_state = SEED
)


LR_param_grid = {
    "C": [10, 100, 1000, 10000],
}

LR_model = train_final_model(cfg, LR_model, LR_param_grid, "Logistic_Regression")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Logistic_Regression
CV Folds       : 5

Hyperparameter Grid:
C                        : [10, 100, 1000, 10000]
Fitting 5 folds for each of 4 candidates, totalling 20 fits

--------------------------------------------------------------------------------
Total time      : 4.29 seconds
--------------------------------------------------------------------------------
Best params for Logistic_Regression:
{'C': 10}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [6]:
################################
# Decision Tree
################################

DT_model = DecisionTreeClassifier(
    class_weight = "balanced",
    max_depth = 6,
    random_state = SEED
)

DT_param_grid = {
    "max_leaf_nodes": [10, 15, 20, 25, 30],
    "min_samples_leaf": [5, 10, 15],
    "criterion": ["gini", "entropy"],
}
DT_model = train_final_model(cfg, DT_model, DT_param_grid, "Decision_Tree")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Decision_Tree
CV Folds       : 5

Hyperparameter Grid:
max_leaf_nodes           : [10, 15, 20, 25, 30]
min_samples_leaf         : [5, 10, 15]
criterion                : ['gini', 'entropy']
Fitting 5 folds for each of 30 candidates, totalling 150 fits

--------------------------------------------------------------------------------
Total time      : 0.28 seconds
--------------------------------------------------------------------------------
Best params for Decision_Tree:
{'criterion': 'entropy', 'max_leaf_nodes': 15, 'min_samples_leaf': 5}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [5]:
################################
# SVM With RBF Kernel
################################

SVM_model = SVC(
    kernel = "rbf",
    decision_function_shape = 'ovr',
    class_weight = "balanced",
    random_state = SEED,
    probability=True
)

SVM_param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1]
}

SVM_model = train_final_model(cfg, SVM_model, SVM_param_grid, "SVM")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : SVM
CV Folds       : 5

Hyperparameter Grid:
C                        : [0.1, 1, 10, 100]
gamma                    : ['scale', 'auto', 0.01, 0.1]
Fitting 5 folds for each of 16 candidates, totalling 80 fits

--------------------------------------------------------------------------------
Total time      : 4.37 seconds
--------------------------------------------------------------------------------
Best params for SVM:
{'C': 10, 'gamma': 'auto'}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [8]:
################################
# Random Forest
################################

RF_model = RandomForestClassifier(
        class_weight = "balanced_subsample",
        random_state = SEED
)


RF_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, None],
    "max_features": ["sqrt", 0.3, 0.4],
    "criterion": ['gini', 'entropy'],
    "min_samples_split": [2, 5],
}  

RF_model = train_final_model(cfg, RF_model, RF_param_grid, "Random_Forest")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Random_Forest
CV Folds       : 5

Hyperparameter Grid:
n_estimators             : [100, 200, 300]
max_depth                : [5, 10, None]
max_features             : ['sqrt', 0.3, 0.4]
criterion                : ['gini', 'entropy']
min_samples_split        : [2, 5]
Fitting 5 folds for each of 108 candidates, totalling 540 fits

--------------------------------------------------------------------------------
Total time      : 22.13 seconds
--------------------------------------------------------------------------------
Best params for Random_Forest:
{'criterion': 'gini', 'max_depth': 10, 'max_features': 0.3, 'min_samples_split': 5, 'n_estimators': 100}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
-----------------------------------------------------------------------------

In [9]:
################################
# XGBoost
################################

XG_model = BalancedXGBClassifier(
        random_state = SEED,
        sampling_method = "uniform",
        objective= "multi:softmax",
        eval_metric="mlogloss",
)

XG_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 0.3],
    "reg_lambda": [1, 5]
}

XG_model = train_final_model(cfg, XG_model, XG_param_grid, "XGBoost")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : XGBoost
CV Folds       : 5

Hyperparameter Grid:
n_estimators             : [100, 300]
max_depth                : [5, 10]
learning_rate            : [0.01, 0.1]
subsample                : [0.8, 1.0]
colsample_bytree         : [0.8, 1.0]
gamma                    : [0, 0.3]
reg_lambda               : [1, 5]
Fitting 5 folds for each of 128 candidates, totalling 640 fits

--------------------------------------------------------------------------------
Total time      : 16.54 seconds
--------------------------------------------------------------------------------
Best params for XGBoost:
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'reg_lambda': 1, 'subsample': 0.8}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------

Ejemplo de carga de un modelo y prediccion de muestras:

In [10]:
# LOAD MODEL

import joblib
saved_artifacts = joblib.load("../output/models/XGBoost_model.pkl")

model = saved_artifacts["model"]
encoder = saved_artifacts["label_encoder"]
model.classes_

array([0, 1, 2, 3, 4])

In [11]:
df = pd.read_csv("../data/processed/model_data.csv")

X = df.drop(columns=["CC", "URB"])
y = df["URB"]

n = 5 

for i in range(n):
    X_row = X.iloc[[i]]  
    y_true = y.iloc[i]

    y_pred_encoded = model.predict(X_row)[0]
    y_pred = encoder.inverse_transform([y_pred_encoded])[0]

    print(f"Fila {i}")
    print(f"  Real      : {y_true}")
    print(f"  Prediccion Encoded: {y_pred_encoded}")
    print(f"  Predicción Decoded: {y_pred}")
    print("-" * 40)


Fila 0
  Real      : 1
  Prediccion Encoded: 0
  Predicción Decoded: 1
----------------------------------------
Fila 1
  Real      : 4
  Prediccion Encoded: 3
  Predicción Decoded: 4
----------------------------------------
Fila 2
  Real      : 4
  Prediccion Encoded: 3
  Predicción Decoded: 4
----------------------------------------
Fila 3
  Real      : 4
  Prediccion Encoded: 3
  Predicción Decoded: 4
----------------------------------------
Fila 4
  Real      : 4
  Prediccion Encoded: 3
  Predicción Decoded: 4
----------------------------------------


In [12]:
model.get_params()

{'random_state': 42,
 'sampling_method': 'uniform',
 'objective': 'multi:softmax',
 'eval_metric': 'mlogloss',
 'colsample_bytree': 0.8,
 'gamma': 0,
 'learning_rate': 0.01,
 'max_depth': 5,
 'n_estimators': 300,
 'reg_lambda': 1,
 'subsample': 0.8}

In [13]:
model.feature_importances_

array([0.04033331, 0.02948242, 0.03127729, 0.0542662 , 0.00672954,
       0.04074478, 0.03921466, 0.14838517, 0.01191737, 0.027504  ,
       0.        , 0.00748485, 0.00764802, 0.00767188, 0.08843508,
       0.08600711, 0.01321572, 0.01172829, 0.05277912, 0.0327234 ,
       0.0196476 , 0.04162645, 0.1343066 , 0.02803902, 0.0388321 ],
      dtype=float32)

In [14]:
model.n_features_in_

25

## 1.2 Mecanismos de interpretabilidad

### 1.2.1 Coeficientes de regresión logística multinomial

El modelo de regresión logística multinomial ofrece una interpretación matemática directa y transparente a través de sus coeficientes. Estos coeficientes establecen la relación entre las variables predictoras y las distintas categorías de la variable objetivo.

Una vez entrenado, el estimador *LogisticRegression* de la librería *scikit-learn* almacena una matriz de coeficientes $\beta$ asociada a cada clase y a cada variable predictora. En este problema, dicha matriz presenta dimensiones $[5,25]$, donde cada fila corresponde a una de las cinco categorías de la variable objetivo y cada columna representa una de las $25$ variables predictoras consideradas en el modelo.

Dado que las variables predictoras son categóricas binarias, la interpretación de los coeficientes se reduce a analizar el efecto que produce pasar de la ausencia de una determinada característica (valor $0$) a su presencia (valor $1$), manteniendo constantes el resto de variables. En términos urbanísticos, esto permite cuantificar el impacto que tiene una determinada característica morfológica, por ejemplo, la presencia de calles sin salida, sobre la probabilidad de que un complejo residencial pertenezca a un determinado grado de cerramiento.

El signo del coeficiente indica la dirección en la que varía la probabilidad de pertenencia a una clase cuando la variable predictora se activa, es decir, cuando pasa de $0$ a $1$:
* $B > 0$. Un coeficiente positivo indica que la presencia de dicha característica incrementa la probabilidad de pertenencia a esa clase en comparación con las demás. Estas variables predictoras pueden interpretarse como factores catalizadores, ya que favorecen la asignación a una categoría concreta.
* $B < 0$. Un coeficiente negativo indica que la presencia de dicha característica reduce la probabilidad de pertenencia a esa clase en comparación con las demás. Estas variables pueden interpretarse como factores inhibidores de una determinada categoría.
* $B\approx0$. Un coeficiente próximo a cero indica que la influencia de esa variable sobre la pertenencia a la clase es prácticamente nula, por lo que su capacidad discriminativa resulta muy limitada.

La magnitud del coeficiente indica cuánto contribuye una variable predictora al valor del *log-odds* asociado a una determinada clase. Dado que la regresión logística multinomial opera en la escala de los log-odds, la relación entre variables y predicción es lineal y aditiva. Por tanto, cada coeficiente se suma o se resta directamente al valor total calculado por el modelo, modificando así la probabilidad estimada de pertenencia a una categoría concreta.

En consecuencia, coeficientes con magnitudes elevadas, ya sean positivas o negativas, reflejan una mayor influencia de la variable sobre la predicción del modelo, mientras que coeficientes cercanos a cero indican una contribución reducida.

La interpretación de la magnitud de los coeficientes depende de la escala de las variables predictoras. En este caso, todas las variables se encuentran representadas en la misma escala binaria, lo que permite comparar directamente los coeficientes entre sí. Este enfoque facilita la identificación de las variables predictoras más relevantes para el modelo en general y más relevantes para cada categoría de la variable objetivo.


Este análisis de interpretabilidad basado en los coeficientes del modelo permite abordar dos niveles de estudio complementarios:
* **Análisis global de importancia de variables**. Permite identificar las variables predictoras más relevantes para el modelo en su conjunto. Para ello, se analiza el valor medio de los coeficientes en valor absoluto a través de todas las clases, de modo que aquellas variables con magnitudes promedio más elevadas se consideran las que ejercen una mayor influencia global sobre las predicciones.
* **Análisis interpretativo por clase**. Permite caracterizar cada categoría de la variable objetivo a partir de las variables cuyos coeficientes tienen un mayor impacto sobre la probabilidad de pertenencia a dicha clase. En este contexto, los coeficientes positivos representan características catalizadoras, mientras que los negativos actúan como factores inhibidores. Este enfoque permite extraer conclusiones interpretables del tipo: "El modelo de regresión logística ha aprendido que el grado de cerramiento “Protegido” se asocia principalmente con la presencia de cámaras de seguridad y la ausencia de calles peatonales".

### 1.2.2 Reglas e importancia de variables del árbol de decisión

Para la interpretación del árbol de decisión se emplean dos mecanismos intrínsecos al propio modelo que permiten explicar su comportamiento: la importancia de variables y la extracción de reglas de decisión.

En un árbol de decisión, la importancia de una variable se define como la reducción acumulada de impureza que dicha variable produce a lo largo de todos los nodos en los que es utilizada para realizar una partición. En otras palabras, una variable es considerada más importante si su uso permite separar los datos en subconjuntos más homogéneos respecto a las clases  de la variable objetivo.  El análisis de la importancia de variables predictoras permite identificar qué características dominan el proceso de decisión del modelo, proporcionando una visión global sobre la información que el árbol utiliza principalmente para realizar la clasificación.

Para comprender cómo el modelo utiliza estas variables en la segmentación del espacio de datos, se recurre a la visualización de la estructura del árbol de decisión. A partir de dicha estructura es posible extraer todas las reglas de decisión asociadas a las hojas terminales.

Cada camino desde el nodo raíz hasta una hoja puede expresarse como una regla del tipo: "SI condicion_1, Y condicion_2, Y..., entonces clase=k".

Estas reglas permiten identificar las combinaciones de variables que el modelo utiliza para asignar cada clase. La fiabilidad de las reglas vendrá determinada por la distribución de clases en la hoja terminal de dicha reglas. Hojas con distribuciones altamente concentradas en una única clase indican decisiones más seguras, mientras que distribuciones más uniformes reflejan una mayor ambigüedad.

Una buena interpretabilidad del árbol de decisión depende por tanto de que las reglas sean semánticamente interpretables desde el dominio del problema y suficientemente discriminativas.

### 1.2.3 SHAP (SHapley Additive exPlanations)

SHAP (SHapley Additive exPlanations), introducido por Lundberg y Lee en 2017, es un marco de trabajo unificado diseñado para interpretar las predicciones generadas por modelos de aprendizaje automático. Esta técnica ha adquirido una especial relevancia en el contexto actual, donde el uso de modelos cada vez más complejos para alcanzar un alto rendimiento predictivo plantea el desafío de comprender por qué dichos modelos toman determinadas decisiones.

Conceptualmente, SHAP pertenece a la categoría de métodos de atribución aditiva de características, ya que explica la salida de un modelo como la suma de las contribuciones individuales de cada variable de entrada. Esto facilita interpretar cuánto contribuye cada característica a desplazar la predicción respecto a la predicción base. La predicción base se define como el valor esperado de la salida del modelo cuando no se conocen las características de la observación. Matemáticamente, esto es la predicción promedio del modelo sobre el conjunto de datos de entrenamiento.

Bajo este enfoque, la explicación se representa mediante una función lineal de variables binarias que indican la presencia o ausencia de cada característica en la predicción. La contribución asignada a cada variable, denominada valor SHAP, se fundamenta en la teoría de juegos cooperativos, especificamente en los valores de Shapley. 

Esta teoría, desarrollada por Lloyd Shapley en 1953, distribuye de manera equitativa las ganancias entre un grupo de jugadores en un juego cooperativo, en función de su contribución marginal al resultado final. En el contexto del aprendizaje automático, el “juego” corresponde a la tarea de predicción para una muestra individual, mientras que los “jugadores” representan las características o variables independientes del modelo.

SHAP se ha consolidado como una de las técnicas de interpretabilidad más utilizadas debido a su papel como marco unificador de una amplia familia de métodos propuestos previamente. Los autores demuestran que enfoques como LIME, DeepLIFT, Layer-Wise Relevance Propagation (LRP), Shapley regression values, Shapley sampling values y Quantitative Input Influence pueden interpretarse bajo un mismo esquema conceptual. Todos ellos constituyen aproximaciones, con distintas estrategias de cálculo, a los valores de Shapley de la teoría de juegos cooperativos. 

SHAP proporciona una interpretación unificada de estos métodos dentro de un mismo fundamento teórico y se posiciona como el único método de atribución aditiva que satisface simultáneamente tres propiedades matemáticas fundamentales:
* Exactitud local (local accuracy): la suma de las contribuciones de las características coincide exactamente con la salida del modelo para cada observación.
* Ausencia (missingness): aquellas características que no están presentes en la entrada no contribuyen a la explicación.
* Consistencia (consistency): si un cambio en el modelo incrementa (o mantiene) la contribución de una característica, su valor de atribución no puede disminuir.

Estas propiedades garantizan matemáticamente una solución única para la atribución de características, algo de lo que carecen los métodos previos. Los autores de SHAP sostienen que el cumplimiento de estas propiedades se alinea con la intuición humana y que su ausencia puede dar lugar a explicaciones percibidas como inconsistentes o poco intuitivas. Para respaldar esta afirmación, presentan un experimento en el que expertos en distintos dominios asignan manualmente la importancia de las variables en diversos problemas. Los resultados muestran un mayor acuerdo entre estas valoraciones humanas y las explicaciones generadas por SHAP en comparación con otros métodos como LIME o DeepLIFT.

Una ventaja adicional de SHAP es su capacidad para proporcionar interpretabilidad tanto a nivel local como global, a diferencia de otros métodos que se limitan exclusivamente a explicaciones globales. Los valores SHAP se calculan a nivel de instancia, lo que permite explicar de forma directa la predicción de observaciones individuales a partir de las características más influyentes en cada caso. Para transitar de esta interpretabilidad local a una visión global, se agregan los valores SHAP a lo largo de todo el conjunto de datos de entrenamiento. Esto da lugar a una medida global de importancia de las variables, definida como el promedio de sus contribuciones locales.

A continuación se describe el fundamento matemático en el que se basa SHAP.

Sea un modelo modelo $f(x)$ que se desea explicar para una observación específica $x$, SHAP establece una función de explicación $g(z')$ a través de un modelo lineal tal que:

$$g(z') = \phi_0 + \sum_{i=1}^{M} \phi_i z'_i$$

donde:
* $z' \in \{0, 1\}^M$ es un vector de presencia de características, denominado "coalición" en teoría de juegos. $z'_i = 1$ significa que la característica $i$ está observada, y $0$ que está ausente.
* $M$ es el número total de características (variables independientes)
* $\phi_i \in \mathbb{R}$ es el valor SHAP de la característica $i$.
* $\phi_0$ es el valor base o predicción esperada.

El valor SHAP ($\phi_i$) para la característica $i$ se define mediante la ecuación clásica de *Shapley*:
$$\phi_i(f, x) = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(M - |S| - 1)!}{M!} \left[ f_x(S \cup \{i\}) - f_x(S) \right]$$

donde:
* $F$ es el conjunto de todas las características.
* $S$ es un subconjunto de características que no incluye a la característica $i$.
* $f_x(S)$ es la función que predice la salida utilizando solo las características presentes en el subconjunto $S$. Esta función se denomina valor de la coalición.
* $\left[ f_x(S \cup \{i\}) - f_x(S) \right]$ es la contribución marginal de la característica $i$ al subconjunto $S$.
* $\frac{|S|!(M - |S| - 1)!}{M!}$ es el factor de ponderación, que representa la probabilidad de que el subconjunto $S$ aparezca en una permutación aleatoria de las características.

El valor SHAP es por tanto el valor promedio ponderado de la contribución marginal de la característica $i$ a través de todas las combinaciones de variables posibles.

Los modelos de aprendizaje automático no están diseñados para recibir subconjuntos de variables como entrada. Por lo tanto, para evaluar $f_x(S)$ cuando faltan características se define el concepto de predicción esperada. Formalmente, el valor de una coalición $S$ se define como la esperanza condicional de la predicción del modelo, dado que conocemos los valores de las características en $S$:

$$f_x(S) = \mathbb{E}_{X_{\setminus S} | X_S = x_S} [f(X)]$$

Donde $X_S = x_S$ son los valores observados de la instancia actual para las características del grupo $S$, y $X_{\setminus S}$ representa las características ausentes. Cuando el subconjunto $S$ es el conjunto vacío ($\emptyset$), significa que no conocemos el valor de ninguna característica de nuestra instancia $x$. En ese escenario:
$$\phi_0 = f_x(\emptyset) = \mathbb{E}[f(X)]$$

Por lo tanto, la predicción esperada ($\phi_0$) se define formalmente como la esperanza matemática de las predicciones del modelo sobre toda la distribución de los datos de entrenamiento. Es la predicción que el modelo haría "a ciegas" si no conociese absolutamente nada sobre la observación actual.

En la práctica calcular la esperanza condicional exacta $\mathbb{E}[f(X) | X_S = x_S]$ es computacionalmente intratable puesto que implica conocer la distribución conjunta de los datos. SHAP resuelve esto mediante dos aproximaciones dependiendo del algoritmo:
* Aproximación de Independencia (KernelSHAP). Asume que las características presentes ($S$) y ausentes ($\setminus S$) son independientes. Bajo esta suposición, la esperanza condicional se convierte en una esperanza marginal. En la práctica esto se implementa utilizando una muestra de fondo (un background dataset) y reemplazando las características ausentes con los valores de esos datos de fondo. Esta ha sido la técnica empleada para calcular los valores SHAP del modelo SVM.
* TreeSHAP para modelos basados en árboles. Para algoritmos como XGBoost y Random Forest, TreeSHAP calcula $\mathbb{E}[f(X) | X_S = x_S]$ de manera exacta y eficiente en tiempo polinomial. Lo logra siguiendo los caminos del árbol de decisión: si una división (split) utiliza una variable ausente (que no está en $S$), el algoritmo sigue ambos caminos del split simultáneamente y pondera las predicciones de las hojas resultantes por la proporción de registros de entrenamiento que fluyen a través de cada rama (citar *Lundberg, S.M., Erion, G., Chen, H. et al. From local explanations to global understanding with explainable AI for trees. Nat Mach Intell*).

Para implementar SHAP, se ha utilizado la librería de Python *shap*, desarrollada por los propios autores del método. En problemas de clasificación multiclase, los valores SHAP se calculan para cada característica y para cada una de las clases de la variable objetivo. De este modo, se interpretará de forma desagregada cómo contribuye cada variable a la salida del modelo en cada clase, cuantificando su influencia específica en la predicción asociada a cada categoría.

# 2. Analisis de interpretabilidad local

TO COMPLETE...